# NBA Polymarket Backtest (Parquet Version) — Colab

In [ ]:
!pip install -q "numpy<2.0.0" "pandas==2.2.2" "scikit-learn==1.5.2" "xgboost==2.1.3" "nba-on-court>=0.2.0,<1.0" "joblib>=1.3" "matplotlib>=3.8" "tqdm>=4.66" "pyarrow"

import os
from google.colab import drive

# Mount Google Drive to save data persistently
drive.mount('/content/drive')
DRIVE_DIR = '/content/drive/MyDrive/nba_backtest_data'
os.makedirs(DRIVE_DIR, exist_ok=True)
print(f"Data directory ready at: {DRIVE_DIR}")

# Clone repository and install local module
REPO_PATH = '/content/nba_bot'
if not os.path.exists(REPO_PATH):
    !git clone https://github.com/cyruslayo/nba_bot.git {REPO_PATH}
else:
    !git -C {REPO_PATH} pull

!pip uninstall -y nba-bot > /dev/null 2>&1 || true
!pip install -e /content/nba_bot --no-deps -q

print('Setup complete')

In [ ]:
import requests
from datetime import datetime, timedelta

def download_pmxt_parquet(start_date, end_date, save_dir):
    current = datetime.strptime(start_date, "%Y-%m-%d")
    end = datetime.strptime(end_date, "%Y-%m-%d")
    
    while current <= end:
        date_str = current.strftime("%Y-%m-%d")
        
        # Loop through 24 hours (00 to 23)
        for hour in range(24):
            hour_str = f"{hour:02d}"
            filename = f"polymarket_orderbook_{date_str}T{hour_str}.parquet"
            url = f"https://archive.pmxt.dev/Polymarket/{filename}"
            save_path = os.path.join(save_dir, filename)
            
            if not os.path.exists(save_path):
                print(f"Downloading {filename}...")
                response = requests.get(url, stream=True)
                
                if response.status_code == 200:
                    with open(save_path, 'wb') as f:
                        for chunk in response.iter_content(chunk_size=8192):
                            f.write(chunk)
                    print(f"Saved to {save_path}")
                else:
                    pass # Skip missing hours without erroring
            else:
                pass # Already exists
                
        current += timedelta(days=1)
    print("Download process finished.")

# Example: Download data for a specific day/week and save it to your Drive (Uncomment to execute)
# download_pmxt_parquet("2024-01-01", "2024-01-02", DRIVE_DIR)

In [ ]:
import pandas as pd
import requests

def search_nba_markets(query="NBA", limit=200, closed=True):
    url = "https://gamma-api.polymarket.com/events"
    params = {
        "slug": query,
        "limit": limit,
        "active": "false" if closed else "true"
    }
    
    response = requests.get(url, params=params)
    if response.status_code != 200:
        print(f"API Error: {response.status_code}")
        return pd.DataFrame()
        
    events = response.json()
    market_data = []
    
    for event in events:
        for market in event.get('markets', []):
            if 'nba' in str(event.get('title', '')).lower() or 'nba' in str(market.get('question', '')).lower():
                market_data.append({
                    'event_title': event.get('title'),
                    'market_question': market.get('question'),
                    'market_id': market.get('id'),
                    'start_date': event.get('startDate')
                })
                
    return pd.DataFrame(market_data)

# Find historical NBA markets
nba_markets_df = search_nba_markets("NBA", limit=200, closed=True)

if not nba_markets_df.empty:
    display(nba_markets_df.head(10))
    MARKET_MAPPING = dict(zip(nba_markets_df['market_id'], nba_markets_df['event_title']))
    print(f"\nFound {len(MARKET_MAPPING)} NBA markets.")
else:
    print("No NBA markets found.")

In [ ]:
import ast
import json
import math
import os
from dataclasses import asdict, dataclass, field
from pathlib import Path
from typing import Any

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from tqdm.auto import tqdm
import pyarrow.parquet as pq

from nba_bot.features import STARTTYPE_FALLBACK, _encode_starttype, build_game_state_rows
from nba_bot.model import load_model, predict_home_win_prob

MODEL_PATH = '/content/drive/MyDrive/nba_bot/xgb_model_t2.pkl'
PMXT_ROOT = Path(DRIVE_DIR)
PMXT_GLOB = '*.parquet'
NBA_SEASONS = [2024]
USE_ADVANCED_FEATURES = True
INITIAL_BANKROLL = 1000.0
LATENCY_BUFFER_MS = 500
COOLDOWN_MS = 1500
MIN_TARGET_DELTA_USDC = 5.0
MIN_DIVERGENCE_RATIO = 0.10
MIN_RESTING_DEPTH_USDC = 50.0
ADVERSE_SELECTION_PENALTY = 0.01
FEE_RATE = 0.02
PER_GAME_BANKROLL_DIVISOR = 5.0
HOLDOUT_PCT = 0.30
CHUNK_SIZE = 100000

# Map your chosen Polymarket IDs to the corresponding NBA Stats game ID here
MARKET_TO_GAME: dict[str, Any] = {
    # '0x_market_id_here': '0022300061' 
}

BACKTEST_STATE_PATH = Path('/content/backtest_state.json')

pd.set_option('display.max_columns', 200)
pd.set_option('display.width', 200)

In [ ]:
def fix_columns(df: pd.DataFrame | None) -> pd.DataFrame | None:
    if df is None: return None
    out = df.copy()
    out.columns = [str(c).upper() for c in out.columns]
    if 'GAME_ID' not in out.columns:
        candidates = [c for c in out.columns if 'GAMEID' in c or 'GAME_ID' in c or 'NBASTATS' in c]
        if candidates: out = out.rename(columns={candidates[0]: 'GAME_ID'})
    return out

def load_data_safe(seasons: list[int], data_type: str) -> pd.DataFrame | None:
    import nba_on_court as noc
    try:
        loaded = noc.load_nba_data(seasons=seasons, data=data_type)
        loaded = fix_columns(loaded)
        if loaded is not None and not loaded.empty: return loaded
    except Exception as exc:
        print(f'Library load failed for {data_type}: {exc}')
    frames = []
    for season in seasons:
        path = Path(f'/content/{data_type}_{season}.tar.xz')
        if path.exists():
            frame = pd.read_csv(path, compression='xz')
            frame = fix_columns(frame)
            frames.append(frame)
    if not frames: return None
    return pd.concat(frames, ignore_index=True)

def load_player_ratings(enabled: bool) -> dict[int, float]:
    if not enabled: return {}
    try:
        from nba_bot.team_stats_cache import refresh_team_stats
        return refresh_team_stats(output_path='/content/team_stats.json')
    except Exception as exc:
        print(f'Player ratings unavailable, falling back to defaults: {exc}')
        return {}

def _candidate_column(columns: list[str], names: list[str]) -> str | None:
    lowered = {str(c).lower(): c for c in columns}
    for name in names:
        if name.lower() in lowered: return lowered[name.lower()]
    for column in columns:
        if any(name.lower() in str(column).lower() for name in names): return column
    return None

def _parse_score_value(value: Any) -> tuple[float, float]:
    text = '' if pd.isna(value) else str(value)
    if ' - ' in text:
        left, right = text.split(' - ', 1)
        try: return float(right), float(left)
        except ValueError: return math.nan, math.nan
    return math.nan, math.nan

def _combine_event_time(frame: pd.DataFrame) -> pd.Series:
    timestamp_col = _candidate_column(list(frame.columns), ['event_time', 'timestamp', 'ts', 'created_at', 'time'])
    if timestamp_col is not None:
        parsed = pd.to_datetime(frame[timestamp_col], utc=True, errors='coerce')
        if parsed.notna().any(): return parsed
    date_col = _candidate_column(list(frame.columns), ['GAME_DATE_EST', 'GAME_DATE', 'DATE'])
    wc_col = _candidate_column(list(frame.columns), ['WCTIMESTRING', 'wall_clock'])
    if date_col is not None and wc_col is not None:
        parsed = pd.to_datetime(frame[date_col].astype(str) + ' ' + frame[wc_col].astype(str), utc=True, errors='coerce')
        if parsed.notna().any(): return parsed
    return pd.to_datetime(pd.Series(np.arange(len(frame))), unit='s', utc=True, errors='coerce')

def _infer_team_ids(game_df: pd.DataFrame) -> tuple[Any, Any]:
    home_team_id, away_team_id = None, None
    for col in ['HOME_TEAM_ID', 'HTEAM_ID', 'HOME_ID']:
        if col in game_df.columns and game_df[col].notna().any():
            home_team_id = game_df[col].dropna().iloc[0]
            break
    for col in ['VISITOR_TEAM_ID', 'VTEAM_ID', 'AWAY_ID']:
        if col in game_df.columns and game_df[col].notna().any():
            away_team_id = game_df[col].dropna().iloc[0]
            break
    if home_team_id is None and 'HOMEDESCRIPTION' in game_df.columns and 'PLAYER1_TEAM_ID' in game_df.columns:
        mask = game_df['HOMEDESCRIPTION'].notna() & game_df['PLAYER1_TEAM_ID'].notna()
        if mask.any(): home_team_id = game_df.loc[mask, 'PLAYER1_TEAM_ID'].iloc[0]
    if away_team_id is None and 'VISITORDESCRIPTION' in game_df.columns and 'PLAYER1_TEAM_ID' in game_df.columns:
        mask = game_df['VISITORDESCRIPTION'].notna() & game_df['PLAYER1_TEAM_ID'].notna()
        if mask.any(): away_team_id = game_df.loc[mask, 'PLAYER1_TEAM_ID'].iloc[0]
    return home_team_id, away_team_id

def prepare_pbp_timeline(df_nbastats: pd.DataFrame, df_pbpstats: pd.DataFrame | None = None, player_ratings: dict[int, float] | None = None) -> pd.DataFrame:
    df = fix_columns(df_nbastats)
    pbp_lookup = fix_columns(df_pbpstats) if df_pbpstats is not None else None
    timeline_rows = []
    
    for game_id, game_df in tqdm(df.groupby('GAME_ID'), desc='Preparing PBP timeline'):
        game_df = game_df.sort_values(['PERIOD', 'PCTIMESTRING'], ascending=[True, False]).copy()
        game_df['event_time'] = _combine_event_time(game_df)
        home_team_id, away_team_id = _infer_team_ids(game_df)
        
        pq_home = float(player_ratings.get(int(home_team_id), 0.0)) if player_ratings and home_team_id is not None else 0.0
        pq_away = float(player_ratings.get(int(away_team_id), 0.0)) if player_ratings and away_team_id is not None else 0.0
        
        pbp_map = {}
        if pbp_lookup is not None and 'GAME_ID' in pbp_lookup.columns:
            game_pbp = pbp_lookup[pbp_lookup['GAME_ID'] == game_id].copy()
            if not game_pbp.empty:
                if 'PCTIMESTRING' not in game_pbp.columns and 'STARTTIME' in game_pbp.columns:
                    game_pbp = game_pbp.rename(columns={'STARTTIME': 'PCTIMESTRING'})
                if 'PCTIMESTRING' in game_pbp.columns:
                    game_pbp['_secs'] = game_pbp['PCTIMESTRING'].astype(str).str.split(':').apply(lambda parts: int(parts[0]) * 60 + int(parts[1]) if len(parts) == 2 else 0)
                    for row in game_pbp.itertuples():
                        period = int(getattr(row, 'PERIOD', 1) or 1)
                        pbp_map.setdefault(period, []).append((int(getattr(row, '_secs', 0) or 0), getattr(row, 'STARTTYPE', None)))
                        
        home_score, away_score = 0.0, 0.0
        for play in game_df.itertuples(index=False):
            parsed_h, parsed_a = _parse_score_value(getattr(play, 'SCORE', None))
            if not math.isnan(parsed_h): home_score = parsed_h
            if not math.isnan(parsed_a): away_score = parsed_a
            
            period = int(getattr(play, 'PERIOD', 1) or 1)
            clock = str(getattr(play, 'PCTIMESTRING', '12:00'))
            clock_parts = clock.split(':')
            secs = int(clock_parts[0]) * 60 + int(clock_parts[1]) if len(clock_parts) == 2 else 720
            time_remaining = float((4 - period) * 720 + secs) if period <= 4 else float(max(0, secs))
            
            st_enc = STARTTYPE_FALLBACK
            for pbp_secs, starttype in pbp_map.get(period, []):
                if abs(pbp_secs - secs) <= 2:
                    st_enc = _encode_starttype(str(starttype))
                    break
                    
            timeline_rows.append({
                'game_id': game_id,
                'event_time': getattr(play, 'event_time'),
                'period': period,
                'clock': clock,
                'home_score': float(home_score),
                'away_score': float(away_score),
                'time_remaining': time_remaining,
                'starttype_encoded': int(st_enc),
                'player_quality_home': pq_home,
                'player_quality_away': pq_away,
            })
            
    timeline = pd.DataFrame(timeline_rows).sort_values(['game_id', 'event_time']).reset_index(drop=True)
    if timeline.empty: raise ValueError('No PBP timeline rows were built')
    timeline['latent_time'] = timeline['event_time'] + pd.to_timedelta(LATENCY_BUFFER_MS, unit='ms')
    tipoff_lookup = timeline.groupby('game_id', as_index=False)['event_time'].min().rename(columns={'event_time': 'tipoff_time'})
    timeline = timeline.merge(tipoff_lookup, on='game_id', how='left')
    return timeline

def _safe_json(value: Any) -> Any:
    if isinstance(value, (list, dict)): return value
    if value is None or (isinstance(value, float) and math.isnan(value)): return None
    if isinstance(value, str):
        text = value.strip()
        if not text: return None
        try: return json.loads(text)
        except Exception:
            try: return ast.literal_eval(text)
            except Exception: return None
    return None

def attach_game_ids(frame: pd.DataFrame, mapping: dict[str, Any]) -> pd.DataFrame:
    out = frame.copy()
    key_col = _candidate_column(list(out.columns), ['market_id', 'slug', 'event_slug', 'market'])
    if key_col is None: raise ValueError('Unable to infer market_id from PMXT data')
    out['game_id'] = out[key_col].astype(str).map(mapping)
    return out[out['game_id'].notna()].copy()

def join_ticks_with_timeline(frame: pd.DataFrame, timeline: pd.DataFrame) -> pd.DataFrame:
    ticks = frame.copy()
    ticks['timestamp'] = pd.to_datetime(ticks['timestamp'], utc=True, errors='coerce')
    ticks = ticks[ticks['timestamp'].notna()].sort_values(['game_id', 'timestamp']).reset_index(drop=True)
    merged = pd.merge_asof(
        ticks,
        timeline.sort_values(['game_id', 'latent_time']),
        left_on='timestamp',
        right_on='latent_time',
        by='game_id',
        direction='backward',
        allow_exact_matches=True,
    )
    return merged[merged['timestamp'] >= merged['tipoff_time']].copy()

In [ ]:
def parse_levels(raw_levels: Any, descending: bool) -> list[tuple[float, float]]:
    payload = _safe_json(raw_levels)
    normalized = []
    for level in payload or []:
        price, size = None, None
        if isinstance(level, dict):
            price = level.get('price') or level.get('px') or level.get('p')
            size = level.get('size') or level.get('sz') or level.get('quantity') or level.get('q')
        elif isinstance(level, (list, tuple)) and len(level) >= 2:
            price, size = level[0], level[1]
        try:
            price, size = float(price), float(size)
        except (TypeError, ValueError):
            continue
        if price <= 0 or price >= 1 or size <= 0: continue
        normalized.append((float(price), float(size)))
    normalized.sort(key=lambda item: item[0], reverse=descending)
    return normalized

class OrderBookState:
    def __init__(self) -> None:
        self.bids: dict[float, float] = {}
        self.asks: dict[float, float] = {}

    def _apply_side(self, side: str, levels: list[tuple[float, float]], replace: bool) -> None:
        book = self.bids if side == 'bids' else self.asks
        if replace: book.clear()
        for price, size in levels:
            if size <= 0: book.pop(price, None)
            else: book[float(price)] = float(size)

    def update(self, record: dict[str, Any]) -> None:
        event_type = str(record.get('event_type') or record.get('type') or '').lower()
        bids = parse_levels(record.get('bids'), descending=True)
        asks = parse_levels(record.get('asks'), descending=False)
        is_snapshot = event_type == 'book_snapshot'
        
        if bids: self._apply_side('bids', bids, replace=is_snapshot)
        elif is_snapshot: self.bids.clear()
        if asks: self._apply_side('asks', asks, replace=is_snapshot)
        elif is_snapshot: self.asks.clear()
            
        for change in _safe_json(record.get('price_change')) or []:
            if isinstance(change, dict):
                side = str(change.get('side') or change.get('book') or '').lower()
                price = change.get('price') or change.get('px')
                size = change.get('size') or change.get('sz') or 0
            elif isinstance(change, (list, tuple)) and len(change) >= 3:
                side, price, size = change[0], change[1], change[2]
            else:
                continue
            try:
                price, size = float(price), float(size)
            except (TypeError, ValueError): continue
            
            target = self.bids if 'bid' in side else self.asks
            if size <= 0: target.pop(price, None)
            else: target[price] = size

    def best_bid(self) -> tuple[float | None, float]:
        if not self.bids: return None, 0.0
        price = max(self.bids)
        return price, float(self.bids[price])

    def best_ask(self) -> tuple[float | None, float]:
        if not self.asks: return None, 0.0
        price = min(self.asks)
        return price, float(self.asks[price])

    def vwap_buy(self, stake_usdc: float) -> tuple[float | None, float]:
        remaining, total_cost, total_shares = float(stake_usdc), 0.0, 0.0
        for price in sorted(self.asks):
            size = float(self.asks[price])
            if size <= 0: continue
            level_cost = float(price) * size
            if level_cost <= remaining:
                total_cost += level_cost
                total_shares += size
                remaining -= level_cost
            else:
                shares = remaining / float(price)
                total_cost += remaining
                total_shares += shares
                remaining = 0.0
                break
        if total_shares <= 0 or remaining > 1e-6: return None, total_shares
        return total_cost / total_shares, total_shares

@dataclass
class MarketPosition:
    market_id: str
    game_id: Any
    yes_shares: float = 0.0
    no_shares: float = 0.0
    yes_cost: float = 0.0
    no_cost: float = 0.0
    realized_pnl: float = 0.0
    cooldown_until: pd.Timestamp | None = None
    settled: bool = False
    settlement_value: float | None = None

@dataclass
class PortfolioState:
    initial_bankroll: float
    cash: float
    realized_pnl: float = 0.0
    positions: dict[str, MarketPosition] = field(default_factory=dict)

def total_bankroll(portfolio: PortfolioState) -> float:
    return float(portfolio.initial_bankroll + portfolio.realized_pnl)

def available_capital(portfolio: PortfolioState) -> float:
    return max(total_bankroll(portfolio) / PER_GAME_BANKROLL_DIVISOR, 0.0)

def fee_adjusted_kelly(probability: float, market_price: float, fee_rate: float = FEE_RATE, fraction: float = 0.25) -> float:
    if not (0 < market_price < 1) or not (0 < probability < 1): return 0.0
    b = (1.0 - market_price) / market_price
    q = 1.0 - probability
    full_kelly = ((probability * (1.0 - fee_rate) * b) - q) / b
    return round(max(full_kelly * fraction, 0.0), 6)

def ensure_position(portfolio: PortfolioState, market_id: str, game_id: Any) -> MarketPosition:
    if market_id not in portfolio.positions:
        portfolio.positions[market_id] = MarketPosition(market_id=market_id, game_id=game_id)
    return portfolio.positions[market_id]

def merge_positions(portfolio: PortfolioState, position: MarketPosition) -> None:
    paired = min(position.yes_shares, position.no_shares)
    if paired <= 0: return
    yes_release_cost = position.yes_cost * (paired / position.yes_shares) if position.yes_shares > 0 else 0.0
    no_release_cost = position.no_cost * (paired / position.no_shares) if position.no_shares > 0 else 0.0
    position.yes_shares -= paired
    position.no_shares -= paired
    position.yes_cost -= yes_release_cost
    position.no_cost -= no_release_cost
    realized = paired - yes_release_cost - no_release_cost
    portfolio.cash += paired
    portfolio.realized_pnl += realized
    position.realized_pnl += realized

def execute_buy(portfolio: PortfolioState, position: MarketPosition, book: OrderBookState, side: str, stake_usdc: float) -> dict[str, float] | None:
    if stake_usdc <= 0: return None
    spend = min(float(stake_usdc), float(portfolio.cash))
    if spend <= 0: return None
    raw_vwap, raw_shares = book.vwap_buy(spend)
    if raw_vwap is None or raw_shares <= 0: return None
    effective_price = min(max(float(raw_vwap) + ADVERSE_SELECTION_PENALTY, 0.001), 0.999)
    shares = spend / effective_price
    portfolio.cash -= spend
    if side == 'YES':
        position.yes_shares += shares
        position.yes_cost += spend
    else:
        position.no_shares += shares
        position.no_cost += spend
    merge_positions(portfolio, position)
    return {'spend': spend, 'price': effective_price, 'shares': shares}

def settle_position(portfolio: PortfolioState, position: MarketPosition, outcome_yes: bool) -> dict[str, float] | None:
    if position.settled: return None
    winning_shares = position.yes_shares if outcome_yes else position.no_shares
    total_cost = position.yes_cost + position.no_cost
    proceeds = float(winning_shares)
    gross_profit = proceeds - total_cost
    fee = max(gross_profit, 0.0) * FEE_RATE
    net_profit = gross_profit - fee
    portfolio.cash += proceeds - fee
    portfolio.realized_pnl += net_profit
    position.realized_pnl += net_profit
    position.yes_shares, position.no_shares, position.yes_cost, position.no_cost = 0.0, 0.0, 0.0, 0.0
    position.settled = True
    position.settlement_value = 1.0 if outcome_yes else 0.0
    return {'proceeds': proceeds, 'gross_profit': gross_profit, 'fee': fee, 'net_profit': net_profit}

In [ ]:
def normalize_pmxt_frame(frame: pd.DataFrame) -> pd.DataFrame:
    out = frame.copy()
    out.columns = [str(c) for c in out.columns]
    
    timestamp_col = _candidate_column(list(out.columns), ['timestamp', 'event_time', 'ts', 'created_at'])
    market_col = _candidate_column(list(out.columns), ['market_id', 'market', 'slug', 'event_slug'])
    event_col = _candidate_column(list(out.columns), ['event_type', 'type'])
    
    if timestamp_col is None or market_col is None:
        raise ValueError('PMXT chunk is missing a timestamp or market identifier column')
        
    out['timestamp'] = pd.to_datetime(out[timestamp_col], utc=True, errors='coerce')
    out['market_id'] = out[market_col].astype(str)
    out['event_type'] = out[event_col].astype(str) if event_col is not None else 'price_change'
    
    for col in ['bids', 'asks', 'price_change', 'resolution']:
        if col not in out.columns: out[col] = None
    return out

def iter_pmxt_chunks(paths: list[Path], chunksize: int = CHUNK_SIZE):
    for path in paths:
        parquet_file = pq.ParquetFile(path)
        for batch in parquet_file.iter_batches(batch_size=chunksize):
            yield path, batch.to_pandas()

def _extract_resolution(record: dict[str, Any]) -> bool | None:
    event_type = str(record.get('event_type') or '').lower()
    if event_type not in {'condition_resolved', 'market_resolution'} and record.get('resolution') in (None, '', {}, []):
        return None
    resolution = record.get('resolution')
    payload = _safe_json(resolution) if not isinstance(resolution, dict) else resolution
    winner = str(payload.get('winner') or payload.get('outcome') or payload.get('result') or '').lower() if isinstance(payload, dict) else str(resolution).lower()
    if winner in {'yes', '1', 'true', 'home'}: return True
    if winner in {'no', '0', 'false', 'away'}: return False
    return None

def evaluate_signal(probability_yes: float, position: MarketPosition, portfolio: PortfolioState, book: OrderBookState) -> tuple[str, float] | None:
    best_ask, ask_size = book.best_ask()
    best_bid, bid_size = book.best_bid()
    if best_ask is None or best_bid is None: return None
    
    if float(best_ask) * float(ask_size) < MIN_RESTING_DEPTH_USDC or float(best_bid) * float(bid_size) < MIN_RESTING_DEPTH_USDC:
        return None
        
    yes_fraction = fee_adjusted_kelly(probability_yes, float(best_ask))
    no_fraction = fee_adjusted_kelly(1.0 - probability_yes, max(min(1.0 - float(best_bid), 0.999), 0.001))
    
    if yes_fraction <= 0 and no_fraction <= 0: return None
    
    if yes_fraction >= no_fraction:
        target = available_capital(portfolio) * yes_fraction
        delta = target - position.yes_cost
        if delta > MIN_TARGET_DELTA_USDC and (delta / max(target, 1.0)) > MIN_DIVERGENCE_RATIO:
            return 'YES', float(delta)
    else:
        target = available_capital(portfolio) * no_fraction
        delta = target - position.no_cost
        if delta > MIN_TARGET_DELTA_USDC and (delta / max(target, 1.0)) > MIN_DIVERGENCE_RATIO:
            return 'NO', float(delta)
    return None

def split_paths(paths: list[Path], holdout_pct: float = HOLDOUT_PCT) -> tuple[list[Path], list[Path]]:
    ordered = sorted(paths)
    if not ordered: return [], []
    cutoff = min(max(1, int(len(ordered) * (1.0 - holdout_pct))), len(ordered))
    return ordered[:cutoff], ordered[cutoff:]

def run_backtest(paths: list[Path], timeline: pd.DataFrame, model: Any, feature_cols: list[str] | None, initial_bankroll: float = INITIAL_BANKROLL) -> dict[str, Any]:
    portfolio = PortfolioState(initial_bankroll=initial_bankroll, cash=initial_bankroll)
    books: dict[str, OrderBookState] = {}
    trades, equity = [], []
    
    for path, chunk in iter_pmxt_chunks(paths):
        joined = join_ticks_with_timeline(attach_game_ids(normalize_pmxt_frame(chunk), MARKET_TO_GAME), timeline)
        
        for record in joined.sort_values('timestamp').to_dict('records'):
            market_id = str(record['market_id'])
            position = ensure_position(portfolio, market_id, record['game_id'])
            book = books.setdefault(market_id, OrderBookState())
            book.update(record)
            
            resolution = _extract_resolution(record)
            if resolution is not None:
                settlement = settle_position(portfolio, position, resolution)
                if settlement:
                    trades.append({'timestamp': record['timestamp'], 'market_id': market_id, 'game_id': record['game_id'], 'action': 'SETTLE_YES' if resolution else 'SETTLE_NO', **settlement})
                equity.append({'timestamp': record['timestamp'], 'equity': portfolio.cash, 'path': str(path)})
                continue
                
            if position.settled or (record.get('time_remaining') is not None and float(record['time_remaining']) <= 0):
                equity.append({'timestamp': record['timestamp'], 'equity': portfolio.cash, 'path': str(path)})
                continue
                
            if position.cooldown_until is not None and pd.Timestamp(record['timestamp']) < position.cooldown_until:
                equity.append({'timestamp': record['timestamp'], 'equity': portfolio.cash, 'path': str(path)})
                continue
                
            adv_ctx = None
            if USE_ADVANCED_FEATURES:
                adv_ctx = {'starttype_encoded': int(record.get('starttype_encoded', STARTTYPE_FALLBACK) or STARTTYPE_FALLBACK), 'player_quality_home': float(record.get('player_quality_home', 0.0) or 0.0), 'player_quality_away': float(record.get('player_quality_away', 0.0) or 0.0)}
                
            prob_yes = predict_home_win_prob(model=model, feature_cols=feature_cols, home_score=int(record['home_score']), away_score=int(record['away_score']), period=int(record['period']), clock=str(record['clock']), advanced_ctx=adv_ctx)
            
            signal = evaluate_signal(prob_yes, position, portfolio, book)
            if signal:
                side, delta = signal
                fill = execute_buy(portfolio, position, book, side, delta)
                if fill:
                    position.cooldown_until = pd.Timestamp(record['timestamp']) + pd.to_timedelta(COOLDOWN_MS, unit='ms')
                    trades.append({'timestamp': record['timestamp'], 'market_id': market_id, 'game_id': record['game_id'], 'action': f'BUY_{side}', 'model_prob_yes': prob_yes, 'cash_after': portfolio.cash, **fill})
            equity.append({'timestamp': record['timestamp'], 'equity': portfolio.cash, 'path': str(path)})
            
    BACKTEST_STATE_PATH.write_text(json.dumps({'initial_bankroll': portfolio.initial_bankroll, 'cash': portfolio.cash, 'realized_pnl': portfolio.realized_pnl, 'positions': {m: asdict(p) for m, p in portfolio.positions.items()}}, default=str, indent=2), encoding='utf-8')
    return {'portfolio': portfolio, 'trades': pd.DataFrame(trades), 'equity_curve': pd.DataFrame(equity), 'state_path': str(BACKTEST_STATE_PATH)}

def plot_equity_curve(result: dict[str, Any], label: str) -> None:
    equity = result['equity_curve'].copy()
    if equity.empty: return
    sampled = equity.iloc[::max(len(equity) // min(2000, len(equity)), 1)].copy()
    plt.figure(figsize=(12, 4))
    plt.plot(sampled['timestamp'], sampled['equity'])
    plt.title(f'Equity Curve — {label}')
    plt.xlabel('Timestamp')
    plt.ylabel('USDC')
    plt.grid(True, alpha=0.3)
    plt.show()

In [ ]:
model, feature_cols = load_model(MODEL_PATH)
if model is None: raise FileNotFoundError(f'Model not found at {MODEL_PATH}')

df_nbastats = load_data_safe(NBA_SEASONS, 'nbastats')
df_pbpstats = load_data_safe(NBA_SEASONS, 'pbpstats') if USE_ADVANCED_FEATURES else None
player_ratings = load_player_ratings(USE_ADVANCED_FEATURES)
timeline = prepare_pbp_timeline(df_nbastats, df_pbpstats, player_ratings)

# Verify PMXT configuration
pmxt_paths = sorted(PMXT_ROOT.glob(PMXT_GLOB))
if not pmxt_paths: raise FileNotFoundError(f'No PMXT files found under {PMXT_ROOT} matching {PMXT_GLOB}')
if not MARKET_TO_GAME: raise ValueError("MARKET_TO_GAME config dictionary is empty! Use Gamma API Explorer to link market IDs to game IDs first.")

# Run Backtest
print(f"Beginning Backtest Iteration on {len(pmxt_paths)} files")
tuning_result = run_backtest(pmxt_paths, timeline, model, feature_cols, initial_bankroll=INITIAL_BANKROLL)

# Display Results
print(f"Ending Balance: {tuning_result['portfolio'].cash:.2f}")
plot_equity_curve(tuning_result, 'full')